In [ ]:
import requests
import tkinter as tk
from tkinter import messagebox
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Arial'

# API CONFIG 
BOOK_SEARCH_URL = "https://openlibrary.org/search.json"
AUTHOR_SEARCH_URL = "https://openlibrary.org/search/authors.json"


# API CALLS
def search_books_api(keyword: str):
    params = {"q": keyword}
    response = requests.get(BOOK_SEARCH_URL, params=params)
    return response.json() if response.status_code == 200 else None


def search_authors_api(keyword: str):
    params = {"q": keyword}
    response = requests.get(AUTHOR_SEARCH_URL, params=params)
    return response.json() if response.status_code == 200 else None


def get_books_by_author_name(author_name):
    params = {"q": author_name}
    response = requests.get(BOOK_SEARCH_URL, params=params)
    return response.json().get("docs", []) if response.status_code == 200 else []


# DATA PROCESSING
def get_book_info(book):
    title = book.get("title", "Unknown Title")
    authors = ", ".join(book.get("author_name", ["Unknown Author"]))
    year = book.get("first_publish_year", "N/A")
    return title, authors, year


def extract_author_info(author):
    name = author.get("name", "Unknown Name")
    top_work = author.get("top_work", "N/A")
    ratings = author.get("ratings_average", "N/A")
    return name, top_work, ratings


# GUI
class LibraryApp:
    def __init__(self, master):
        self.master = master
        master.title("OPENLIBRARY SEARCH TOOL")

        # Main Title
        self.heading = tk.Label(master, text="OpenLibrary Search Tool", font=('Arial', 18, 'bold'))
        self.heading.grid(row=0, column=0, columnspan=3, pady=10)

        # Input
        tk.Label(master, text="Search Keyword:", font=('Arial', 12)).grid(row=1, column=0, sticky="e", padx=5, pady=5)

        self.query_entry = tk.Entry(master, width=40, font=('Arial', 12))
        self.query_entry.grid(row=1, column=1, padx=5, pady=5)

        # Buttons
        self.book_button = tk.Button(master, text="Search Books", width=15, command=self.search_books)
        self.book_button.grid(row=1, column=2, padx=5)

        self.author_button = tk.Button(master, text="Search Author", width=15, command=self.search_author)
        self.author_button.grid(row=2, column=2, padx=5, pady=5)

        # Output Box
        self.output_box = tk.Text(master, width=90, height=25, wrap="word", font=('Arial', 11))
        self.output_box.grid(row=3, column=0, columnspan=3, padx=10, pady=10)

        self.output_box.tag_config("primary_heading", font=('Arial', 20, "bold"))
        self.output_box.tag_config("subheading", font=('Arial', 14, "bold"))
        self.output_box.tag_config("normal", font=('Arial', 11))

    # BOOK SEARCH FUNCTION
    def search_books(self):
        keyword = self.query_entry.get().strip()

        if not keyword:
            messagebox.showwarning("Input Error", "Please enter a keyword.")
            return

        data = search_books_api(keyword)
        if data is None:
            messagebox.showerror("API Error", "Could not retrieve book information.")
            return

        results = data.get("docs", [])
        self.output_box.delete("1.0", tk.END)

        if not results:
            self.output_box.insert(tk.END, "No books found.")
            return

        for i, book in enumerate(results[:50], start=1):
            title, authors, year = get_book_info(book)

            self.output_box.insert(tk.END, f"Book {i}\n", "subheading")
            self.output_box.insert(tk.END, f"Title: {title}\n", "normal")
            self.output_box.insert(tk.END, f"Author(s): {authors}\n", "normal")
            self.output_box.insert(tk.END, f"Published: {year}\n", "normal")
            self.output_box.insert(tk.END, "-" * 70 + "\n\n", "normal")

    # AUTHOR SEARCH FUNCTION
    def search_author(self):
        keyword = self.query_entry.get().strip()

        if not keyword:
            messagebox.showwarning("Input Error", "Please enter a keyword.")
            return

        author_data = search_authors_api(keyword)
        self.output_box.delete("1.0", tk.END)

        if not author_data:
            self.output_box.insert(tk.END, "Error retrieving author data.\n", "normal")
            return

        results = author_data.get("docs", [])
        if not results:
            self.output_box.insert(tk.END, "No authors found.\n", "normal")
            return

        author_info = results[0]  # first author match
        name, top_work, ratings = extract_author_info(author_info)

        # Output
        self.output_box.insert(tk.END, f"{name}\n", "primary_heading")
        self.output_box.insert(tk.END, "-" * 60 + "\n", "normal")

        self.output_box.insert(tk.END, "Famous Work: ", "subheading")
        self.output_box.insert(tk.END, f"{top_work}\n", "normal")

        self.output_box.insert(tk.END, "Rating Score: ", "subheading")
        self.output_box.insert(tk.END, f"{ratings}\n\n", "normal")

        # Books written by this author
        books = get_books_by_author_name(name)

        self.output_box.insert(tk.END, "\nOther Books:\n", "subheading")
        self.output_box.insert(tk.END, "-" * 40 + "\n", "normal")

        if not books:
            self.output_box.insert(tk.END, "No books were found for this author.\n", "normal")
            return

        for book in books[:10]:
            title = book.get("title", "Unknown Title")
            year = book.get("first_publish_year", "Unknown")
            self.output_box.insert(tk.END, f"• {title} ({year})\n", "normal")


In [ ]:
#RUN APPLICATION
root = tk.Tk()
app = LibraryApp(root)
root.mainloop()